#### Copyright 2017 Google LLC.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

 # Introducción a las redes neuronales

 **Objetivos de aprendizaje:**
  * definir una red neuronal (RN) y sus capas ocultas a través de la clase `DNNRegressor` de TensorFlow
  * entrenar una red neuronal para aprender no linealidades en un conjunto de datos y lograr un mejor rendimiento que un modelo de regresión lineal

 En los ejercicios anteriores, usamos atributos sintéticos para ayudar a nuestro modelo a incorporar no linealidades.

Había un conjunto de no linealidades importante en torno a latitud y longitud, pero pueden existir otros.

Por el momento también volveremos a una tarea de regresión estándar, en lugar de la tarea de regresión logística del ejercicio anterior. Esto significa que prediremos `median_house_value` directamente.

 ## Preparación

Primero, carguemos y preparemos los datos.

In [35]:
from __future__ import print_function

import math
import logging

import tensorflow as tf
import tensorflow.compat.v1 as tf_v1

tf_v1.disable_v2_behavior()

from IPython import display
from matplotlib import cm
from matplotlib import gridspec
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics


# Configurar logs de TensorFlow (reemplazo de tf.logging)
logging.getLogger("tensorflow").setLevel(logging.ERROR)

pd.options.display.max_rows = 10
pd.options.display.float_format = '{:.1f}'.format

# Cargar datos
california_housing_dataframe = pd.read_csv(
    "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv",
    sep=","
)

# Mezclar datos
california_housing_dataframe = california_housing_dataframe.reindex(
    np.random.permutation(california_housing_dataframe.index)
)

In [ ]:
def preprocess_features(california_housing_dataframe):
  """Prepares input features from California housing data set.

  Args:
    california_housing_dataframe: A Pandas DataFrame expected to contain data
      from the California housing data set.
  Returns:
    A DataFrame that contains the features to be used for the model, including
    synthetic features.
  """
  selected_features = california_housing_dataframe[
    ["latitude",
     "longitude",
     "housing_median_age",
     "total_rooms",
     "total_bedrooms",
     "population",
     "households",
     "median_income"]]
  processed_features = selected_features.copy()
  # Create a synthetic feature.
  processed_features["rooms_per_person"] = (
    california_housing_dataframe["total_rooms"] /
    california_housing_dataframe["population"])
  return processed_features

def preprocess_targets(california_housing_dataframe):
  """Prepares target features (i.e., labels) from California housing data set.

  Args:
    california_housing_dataframe: A Pandas DataFrame expected to contain data
      from the California housing data set.
  Returns:
    A DataFrame that contains the target feature.
  """
  output_targets = pd.DataFrame()
  # Scale the target to be in units of thousands of dollars.
  output_targets["median_house_value"] = (
    california_housing_dataframe["median_house_value"] / 1000.0)
  return output_targets

In [24]:
# Choose the first 12000 (out of 17000) examples for training.
training_examples = preprocess_features(california_housing_dataframe.head(12000))
training_targets = preprocess_targets(california_housing_dataframe.head(12000))

# Choose the last 5000 (out of 17000) examples for validation.
validation_examples = preprocess_features(california_housing_dataframe.tail(5000))
validation_targets = preprocess_targets(california_housing_dataframe.tail(5000))

# Double-check that we've done the right thing.
print("Training examples summary:")
display.display(training_examples.describe())
print("Validation examples summary:")
display.display(validation_examples.describe())

print("Training targets summary:")
display.display(training_targets.describe())
print("Validation targets summary:")
display.display(validation_targets.describe())

Training examples summary:


,latitude,longitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_person
count,12000.0,12000.0,12000.0,12000.0,12000.0,12000.0,12000.0,12000.0,12000.0
mean,35.6,-119.6,28.5,2631.0,538.9,1426.2,500.3,3.9,2.0
std,2.1,2.0,12.5,2125.5,417.2,1109.6,379.8,1.9,1.2
min,32.5,-124.3,1.0,8.0,1.0,3.0,1.0,0.5,0.1
25%,33.9,-121.8,18.0,1458.8,297.0,791.0,282.0,2.6,1.5
50%,34.2,-118.5,29.0,2125.0,434.0,1169.0,409.0,3.5,1.9
75%,37.7,-118.0,37.0,3144.5,649.0,1724.0,605.0,4.7,2.3
max,42.0,-114.3,52.0,37937.0,6445.0,28566.0,6082.0,15.0,55.2


Validation examples summary:


,latitude,longitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,rooms_per_person
count,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0,5000.0
mean,35.6,-119.5,28.7,2674.1,540.7,1437.8,503.3,3.9,2.0
std,2.1,2.0,12.7,2305.3,431.8,1234.9,395.7,1.9,1.2
min,32.5,-124.3,2.0,2.0,2.0,6.0,2.0,0.5,0.0
25%,33.9,-121.8,18.0,1468.0,296.0,788.0,282.0,2.6,1.5
50%,34.2,-118.5,29.0,2138.0,433.0,1163.0,409.0,3.6,1.9
75%,37.7,-118.0,37.0,3161.2,648.0,1715.5,606.0,4.8,2.3
max,42.0,-114.5,52.0,32054.0,5290.0,35682.0,5050.0,15.0,52.0


Training targets summary:


,median_house_value
count,12000.0
mean,206.4
std,115.4
min,15.0
25%,119.4
50%,178.6
75%,263.3
max,500.0


Validation targets summary:


,median_house_value
count,5000.0
mean,209.4
std,117.3
min,15.0
25%,119.5
50%,183.6
75%,269.0
max,500.0


 ## Creación de una red neuronal

La RN se define a través de la clase [DNNRegressor](https://www.tensorflow.org/api_docs/python/tf/estimator/DNNRegressor).

Usa **`hidden_units`** para definir la estructura de la RN. El argumento `hidden_units` proporciona una lista de enteros, en la que cada entero corresponde a una capa oculta e indica el número de nodos en ella. Por ejemplo, considera la siguiente asignación:

`hidden_units=[3,10]`

La asignación anterior especifica una red neuronal con dos capas ocultas:

* La primera capa oculta contiene 3 nodos.
* La segunda capa oculta contiene 10 nodos.

Si quisiéramos agregar más capas, incorporaríamos más enteros a la lista. Por ejemplo, `hidden_units=[10,20,30,40]` crearía cuatro capas con diez, veinte, treinta y cuarenta unidades, respectivamente.

De forma predeterminada, todas las capas ocultas usarán activación ReLu y estarán totalmente conectadas.

In [25]:
def construct_feature_columns(input_features):
  """Construct the TensorFlow Feature Columns.

  Args:
    input_features: The names of the numerical input features to use.
  Returns:
    A set of feature columns
  """
  return set([tf.feature_column.numeric_column(my_feature)
              for my_feature in input_features])

In [41]:
def my_input_fn(features, targets, batch_size=1, shuffle=True, num_epochs=None):
    """Trains a neural net regression model.

    Args:
      features: pandas DataFrame of features
      targets: pandas DataFrame of targets
      batch_size: Size of batches to be passed to the model
      shuffle: True or False. Whether to shuffle the data.
      num_epochs: Number of epochs for which data should be repeated. None = repeat indefinitely
    Returns:
      Tuple of (features, labels) for next data batch
    """

    # Convert pandas data into a dict of np arrays.
    features = {key:np.array(value) for key,value in dict(features).items()}

    # Construct a dataset, and configure batching/repeating.
    ds = tf.data.Dataset.from_tensor_slices((features,targets)) # warning: 2GB limit
    ds = ds.batch(batch_size).repeat(num_epochs)

    # Shuffle the data, if specified.
    if shuffle:
      ds = ds.shuffle(10000)

    # Return the next batch of data.
    features, labels = ds.make_one_shot_iterator().get_next()
    return features, labels

In [44]:

def train_nn_regression_model(
    learning_rate,
    steps,
    batch_size,
    hidden_units,
    training_examples,
    training_targets,
    validation_examples,
    validation_targets):

  """Trains a neural network regression model."""

  periods = 10
  steps_per_period = int(steps / periods)

  # 🔥 Modelo Keras (reemplazo moderno de estimator)
  model = tf.keras.Sequential()

  # Capas ocultas
  for units in hidden_units:
    model.add(tf.keras.layers.Dense(units, activation='relu'))

  # Capa de salida
  model.add(tf.keras.layers.Dense(1))

  # Compilación
  model.compile(
      optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
      loss='mse',
      metrics=[tf.keras.metrics.RootMeanSquaredError()]
  )

  print("Training model...")
  print("RMSE (on training data):")

  training_rmse = []
  validation_rmse = []

  # Entrenamiento por periodos
  for period in range(periods):

    history = model.fit(
        training_examples,
        training_targets["median_house_value"],
        validation_data=(validation_examples, validation_targets["median_house_value"]),
        epochs=1,
        batch_size=batch_size,
        verbose=0
    )

    training_root_mean_squared_error = history.history["root_mean_squared_error"][0]
    validation_root_mean_squared_error = history.history["val_root_mean_squared_error"][0]

    print("  period %02d : %0.2f" % (period, training_root_mean_squared_error))

    training_rmse.append(training_root_mean_squared_error)
    validation_rmse.append(validation_root_mean_squared_error)

  print("Model training finished.")

  # 📈 Gráfica
  plt.ylabel("RMSE")
  plt.xlabel("Periods")
  plt.title("Root Mean Squared Error vs. Periods")
  plt.tight_layout()
  plt.plot(training_rmse, label="training")
  plt.plot(validation_rmse, label="validation")
  plt.legend()

  print("Final RMSE (on training data):   %0.2f" % training_root_mean_squared_error)
  print("Final RMSE (on validation data): %0.2f" % validation_root_mean_squared_error)

  return model


 ## Tarea 1: Entrenar un modelo de RN
**Ajusta los hiperparámetros con la intención de disminuir el RMSE por debajo de 110.**

Ejecuta el siguiente bloque para entrenar un modelo de RN.  

Recuerda que, en el ejercicio de regresión lineal, con muchos atributos, un RMSE de aproximadamente 110 era bastante bueno. Intentaremos mejorarlo.
Tu tarea aquí es modificar las distintas configuraciones de aprendizaje para mejorar la exactitud en los datos de validación.

El sobreajuste es un verdadero riesgo potencial para las RN. Puedes observar la brecha entre la pérdida en los datos de entrenamiento y la pérdida en los datos de validación para determinar si tu modelo está comenzando a tener un sobreajuste. Si la brecha comienza a crecer, por lo general es un indicador seguro de sobreajuste.

Debido al número de las diferentes configuraciones posibles, se recomienda enfáticamente tomar nota de cada prueba como guía para el proceso de desarrollo.

Además, cuando obtengas una buena configuración, prueba ejecutarla varias veces y observa qué tan constante es el resultado. Las ponderaciones de RN suelen inicializarse con valores aleatorios pequeños, de manera que debes observar las diferencias entre una ejecución y otra.



In [51]:
pip install tensorflow-estimator

In [52]:
dnn_regressor = train_nn_regression_model(
    learning_rate=0.01,
    steps=500,
    batch_size=10,
    hidden_units=[10, 2],
    training_examples=training_examples,
    training_targets=training_targets,
    validation_examples=validation_examples,
    validation_targets=validation_targets)

AttributeError: module 'tensorflow' has no attribute 'estimator'

 ### Solución

Haz clic más abajo para ver una solución posible.

 **NOTA:** Esta selección de parámetros es algo arbitraria. Aquí probamos combinaciones que son cada vez más complejas, combinadas con el entrenamiento durante más tiempo, hasta que el error se encuentra por debajo de nuestro objetivo. Esta no es de ningún modo la mejor combinación; otras pueden alcanzar un RMSE incluso más bajo. Si intentas encontrar el modelo que pueda alcanzar el mejor error, deberás usar un proceso más riguroso, como una búsqueda de parámetros.

In [43]:
dnn_regressor = train_nn_regression_model(
    learning_rate=0.001,
    steps=2000,
    batch_size=100,
    hidden_units=[10, 10],
    training_examples=training_examples,
    training_targets=training_targets,
    validation_examples=validation_examples,
    validation_targets=validation_targets)

AttributeError: module 'tensorflow.compat.v1' has no attribute 'estimator'

 ## Tarea 2: Evaluar los datos de prueba

**Confirma que los resultados de tu rendimiento de validación respalden los datos de prueba.**

Una vez que tengas un modelo con el que estés satisfecho, evalúalo con los datos de prueba para compararlos con el rendimiento de validación.

Recuerda que el conjunto de datos de prueba está ubicado [aquí](https://download.mlcc.google.com/mledu-datasets/california_housing_test.csv).

In [ ]:
california_housing_test_data = pd.read_csv("https://download.mlcc.google.com/mledu-datasets/california_housing_test.csv", sep=",")

# YOUR CODE HERE

 ### Solución

Haz clic más abajo para ver una solución posible.

 De manera similar a lo que hace el código de la parte superior, tenemos que cargar el archivo de datos adecuado, procesarlo previamente y llamar a predict y mean_squared_error.

Ten en cuenta que no necesitamos aleatorizar los datos de prueba, dado que usaremos todos los registros.

In [42]:
california_housing_test_data = pd.read_csv("https://download.mlcc.google.com/mledu-datasets/california_housing_test.csv", sep=",")

test_examples = preprocess_features(california_housing_test_data)
test_targets = preprocess_targets(california_housing_test_data)

predict_testing_input_fn = lambda: my_input_fn(test_examples,
                                               test_targets["median_house_value"],
                                               num_epochs=1,
                                               shuffle=False)

test_predictions = dnn_regressor.predict(input_fn=predict_testing_input_fn)
test_predictions = np.array([item['predictions'][0] for item in test_predictions])

root_mean_squared_error = math.sqrt(
    metrics.mean_squared_error(test_predictions, test_targets))

print("Final RMSE (on test data): %0.2f" % root_mean_squared_error)

NameError: name 'dnn_regressor' is not defined